Sample:
preferences = [
    "Funny mystery",
    "No horror",
    "Under 2 hours",
    "Like Interstellar"
]

In [4]:
# pip install sentence-transformers

In [5]:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")
# text="A movie with me"
# embedding = model.encode(text)
# print(type(embedding))
# print(embedding)

# sentence1 = "A team of astronauts travel through space."
# sentence2 = "People explore the universe."
# sentence3 = "A romantic love story."
# from sentence_transformers import util
# embedding1 = model.encode(sentence1)
# embedding2 = model.encode(sentence2)
# embedding3 = model.encode(sentence3)
# score12 = util.cos_sim(embedding1, embedding2)
# score13 = util.cos_sim(embedding1, embedding3)
# print(score12)
# print(score13)

In [6]:
import pandas as pd
import numpy as np
import json

df = pd.read_csv("TMDB_movie_dataset.csv")
# df=df.head(10000)
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,original_title,overview,popularity,poster_path,tagline,genres,production_companies,production_countries,spoken_languages,keywords
0,27205,Inception,8.364,34495,Released,2010-07-15,825532764,148,False,/8ZTVqvKDQ8emSGUEMjsS4yHAwrp.jpg,...,Inception,"Cobb, a skilled thief who commits corporate es...",83.952,/oYuLEt3zVCKq57qu2F8dT7NIa6f.jpg,Your mind is the scene of the crime.,"Action, Science Fiction, Adventure","Legendary Pictures, Syncopy, Warner Bros. Pict...","United Kingdom, United States of America","English, French, Japanese, Swahili","rescue, mission, dream, airplane, paris, franc..."
1,157336,Interstellar,8.417,32571,Released,2014-11-05,701729206,169,False,/pbrkL804c8yAv3zBZR4QPEafpAR.jpg,...,Interstellar,The adventures of a group of explorers who mak...,140.241,/gEU2QniE6E77NI6lCU6MxlNBvIx.jpg,Mankind was born on Earth. It was never meant ...,"Adventure, Drama, Science Fiction","Legendary Pictures, Syncopy, Lynda Obst Produc...","United Kingdom, United States of America",English,"rescue, future, spacecraft, race against time,..."
2,155,The Dark Knight,8.512,30619,Released,2008-07-16,1004558444,152,False,/nMKdUUepR0i5zn0y1T4CsSB5chy.jpg,...,The Dark Knight,Batman raises the stakes in his war on crime. ...,130.643,/qJ2tW6WMUDux911r6m7haRef0WH.jpg,Welcome to a world without rules.,"Drama, Action, Crime, Thriller","DC Comics, Legendary Pictures, Syncopy, Isobel...","United Kingdom, United States of America","English, Mandarin","joker, sadism, chaos, secret identity, crime f..."
3,19995,Avatar,7.573,29815,Released,2009-12-15,2923706026,162,False,/vL5LR6WdxWPjLPFRLe133jXWsh5.jpg,...,Avatar,"In the 22nd century, a paraplegic Marine is di...",79.932,/kyeqWdyUXW608qlYkRqosgbbJyK.jpg,Enter the world of Pandora.,"Action, Adventure, Fantasy, Science Fiction","Dune Entertainment, Lightstorm Entertainment, ...","United States of America, United Kingdom","English, Spanish","future, society, culture clash, space travel, ..."
4,24428,The Avengers,7.710,29166,Released,2012-04-25,1518815515,143,False,/9BBTo63ANSmhC4e6r62OJFuK2GL.jpg,...,The Avengers,When an unexpected enemy emerges and threatens...,98.082,/RYMX2wcKCBAr24UyPD7xwmjaTn.jpg,Some assembly required.,"Science Fiction, Action, Adventure",Marvel Studios,United States of America,"English, Hindi, Russian","new york city, superhero, shield, based on com..."


In [7]:
df.columns.tolist()

['id',
 'title',
 'vote_average',
 'vote_count',
 'status',
 'release_date',
 'revenue',
 'runtime',
 'adult',
 'backdrop_path',
 'budget',
 'homepage',
 'imdb_id',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'tagline',
 'genres',
 'production_companies',
 'production_countries',
 'spoken_languages',
 'keywords']

In [8]:
#Not required features-> adult,status,revenue,budget,original_language,original_title,poster_path,production_companies,production_countries,spoken_languages
#Important->title,release_date,runtime,overview,popularity,tagline,genres,keywords

df = df.drop(columns=["backdrop_path","adult","status","revenue","budget","homepage","original_language","original_title","poster_path","production_companies","production_countries","spoken_languages"])
df.columns.to_list()

['id',
 'title',
 'vote_average',
 'vote_count',
 'release_date',
 'runtime',
 'imdb_id',
 'overview',
 'popularity',
 'tagline',
 'genres',
 'keywords']

In [9]:
cols = ["title", "overview", "tagline", "genres", "keywords"]
df[cols] = df[cols].fillna("")
df = df.dropna(subset=["title"])

df["movie_text"] = (
    df["title"] + " " +
    df["overview"] + " " +
    df["tagline"] + " " +
    df["genres"] + " " +
    df["keywords"]
)

print(df["movie_text"].iloc[0])

Inception Cobb, a skilled thief who commits corporate espionage by infiltrating the subconscious of his targets is offered a chance to regain his old life as payment for a task considered to be impossible: "inception", the implantation of another person's idea into a target's subconscious. Your mind is the scene of the crime. Action, Science Fiction, Adventure rescue, mission, dream, airplane, paris, france, virtual reality, kidnapping, philosophy, spy, allegory, manipulation, car crash, heist, memory, architecture, los angeles, california, dream world, subconscious


## Semantic Search

In [ ]:
from sentence_transformers import SentenceTransformer
model=SentenceTransformer("all-MiniLM-L6-v2")

import os

if os.path.exists("movie_embeddings.npy"):
    print("Loading saved embeddings...")
    embeddings = np.load("movie_embeddings.npy")
    if len(embeddings) != len(df):
        print("Cached embeddings don't match current df, regenerating...")
        movie_texts = df["movie_text"].tolist()

        embeddings = model.encode(
            movie_texts,
            batch_size=64,
            show_progress_bar=True
        )

        np.save("movie_embeddings.npy", embeddings)
else:
    print("Generating embeddings...")
    movie_texts = df["movie_text"].tolist()

    embeddings = model.encode(
        movie_texts,
        batch_size=64,
        show_progress_bar=True
    )

    np.save("movie_embeddings.npy", embeddings)
# The order of the embeddings is the same as the order of the DataFrame.
    

In [ ]:
C = df["vote_average"].mean()
m = df["vote_count"].quantile(0.90)  # only movies above this vote_count are trusted at near-full weight

def weighted_rating(row, m=m, C=C):
    v = row["vote_count"]
    R = row["vote_average"]
    return (v / (v + m)) * R + (m / (v + m)) * C

df["weighted_rating"] = df.apply(weighted_rating, axis=1)

# normalize weighted_rating to 0-1 so it's on the same scale as cosine similarity
_wr_min = df["weighted_rating"].min()
_wr_max = df["weighted_rating"].max()
if _wr_max > _wr_min:
    df["weighted_rating_norm"] = (df["weighted_rating"] - _wr_min) / (_wr_max - _wr_min)
else:
    df["weighted_rating_norm"] = 0.0

df[["title", "vote_average", "vote_count", "weighted_rating"]].sort_values("weighted_rating", ascending=False).head(10)

In [12]:
from sentence_transformers import util
import torch

alpha = 0.7  # weight on semantic similarity; (1 - alpha) goes to weighted_rating

user_input = input("Enter your preference: ")

user_embedding = model.encode(user_input, convert_to_tensor=True)
movie_embeddings = torch.tensor(embeddings)

similarities = util.cos_sim(user_embedding, movie_embeddings)[0]

final_scores = alpha * similarities.numpy() + (1 - alpha) * df["weighted_rating_norm"].values

top_indices = final_scores.argsort()[::-1]

for idx in top_indices[:10]:
    print(f"{final_scores[idx]:.4f} (sim={similarities[idx]:.4f}, rating={df.iloc[idx]['weighted_rating']:.2f}) - {df.iloc[idx]['title']}")

0.7336 (sim=0.7101, rating=7.56) - Interstellar: Nolan's Odyssey
0.6992 (sim=0.6176, rating=8.42) - Interstellar
0.6926 (sim=0.9565, rating=1.52) - interstellar
0.6637 (sim=0.6853, rating=6.07) - The Science of Interstellar
0.6166 (sim=0.6563, rating=5.31) - Inside 'Interstellar'
0.5952 (sim=0.5222, rating=7.36) - National Geographic: Journey to the Edge of the Universe
0.5917 (sim=0.5431, rating=6.85) - Starman
0.5738 (sim=0.4219, rating=8.74) - EXO PLANET #2 The EXO'luxion in Japan
0.5738 (sim=0.5254, rating=6.69) - Another Earth
0.5733 (sim=0.5047, rating=7.09) - Lightyear


## Search for similar movie

In [ ]:
movie_title_input = input("Enter movie title").lower()

title_to_index = {}
for i, t in zip(df.index, df["title"]):
    t_lower = t.lower()
    if t_lower not in title_to_index:
        title_to_index[t_lower] = i

if movie_title_input not in title_to_index:
    print(f"'{movie_title_input}' not found in the dataset. Check spelling or try another title.")
else:
    match_idx = title_to_index[movie_title_input]
    search_movie_embedding = embeddings[df.index.get_loc(match_idx)]

    similar_movies = util.cos_sim(search_movie_embedding, embeddings)[0]
    top_indices = similar_movies.argsort(descending=True)

    final_scores = alpha * similar_movies.numpy() + (1 - alpha) * df["weighted_rating_norm"].values
    top_indices = final_scores.argsort()[::-1]
    count = 0
    for idx in top_indices:
        idx = int(idx)
        if df.iloc[idx]["title"].lower() == movie_title_input:
            continue
        print(f"{similar_movies[idx]:.4f} - {df.iloc[idx]['title']}")
        count += 1
        if count == 10:
            break

## Because you liked (Hybrid Search)

In [ ]:
from sklearn.neighbors import NearestNeighbors

movie_df = pd.read_csv("movie.csv")
rating_df = pd.read_csv("rating.csv")

In [27]:
from scipy.sparse import csr_matrix

user_to_idx = {u: i for i, u in enumerate(rating_df["userId"].unique())}
movie_to_idx = {m: i for i, m in enumerate(rating_df["movieId"].unique())}
idx_to_movie = {i: m for m, i in movie_to_idx.items()}

rows = rating_df["movieId"].map(movie_to_idx)
cols = rating_df["userId"].map(user_to_idx)

movie_user_matrix = csr_matrix(
    (rating_df["rating"], (rows, cols))
)

model = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

model.fit(movie_user_matrix)

,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [28]:
my_ratings = {
    "Interstellar (2014)": 5,
    "Inception (2010)": 5,
    "Titanic (1997)": 2
}

In [ ]:
def recommend_movies(my_ratings, top_n=10, neighbours=20):

    # Final weighted score of every candidate movie
    scores = {}

    # Sum of similarities (used for normalization)
    similarity_sum = {}

    # Convert watched movie titles into a set
    watched_movies = set(my_ratings.keys())

    # Process every movie rated by the user
    for movie_title, user_rating in my_ratings.items():

        # Find that movie in MovieLens
        movie_row = movie_df[movie_df["title"] == movie_title]

        if movie_row.empty:
            print(f"{movie_title} not found.")
            continue

        movie_id = movie_row.iloc[0]["movieId"]

        # Convert movieId into matrix index
        movie_index = movie_to_idx[movie_id]

        # Find nearest neighbours
        distances, indices = model.kneighbors(
            movie_user_matrix[movie_index],
            n_neighbors=neighbours
        )

        # Skip first neighbour (itself)
        for distance, neighbour_index in zip(
            distances[0][1:],
            indices[0][1:]
        ):

            similarity = 1 - distance #KNN returns distance (0 is the closest)

            neighbour_movie_id = idx_to_movie[neighbour_index]

            neighbour_title = movie_df.loc[
                movie_df["movieId"] == neighbour_movie_id,
                "title"
            ].values[0]

            # Don't recommend movies already watched
            if neighbour_title in watched_movies:
                continue

            # Weighted contribution
            scores[neighbour_title] = (
                scores.get(neighbour_title, 0)
                + similarity * user_rating
            )

            # Save total similarity
            similarity_sum[neighbour_title] = (
                similarity_sum.get(neighbour_title, 0)
                + similarity
            )

    # Normalize scores
    recommendations = []

    for movie in scores:

        final_score = (
            scores[movie] /
            similarity_sum[movie]
        )

        recommendations.append(
            (movie, final_score)
        )

    recommendations.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return recommendations[:top_n]

In [30]:
recommendations = recommend_movies(my_ratings)

for movie, score in recommendations:
    print(f"{movie:40} {score:.3f}")

X-Men: Days of Future Past (2014)        5.000
Dark Knight Rises, The (2012)            5.000
Prestige, The (2006)                     5.000
King's Speech, The (2010)                5.000
Gone Girl (2014)                         5.000
Edge of Tomorrow (2014)                  5.000
Birdman (2014)                           5.000
Grand Budapest Hotel, The (2014)         5.000
Gravity (2013)                           5.000
The Imitation Game (2014)                5.000
